<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/GPT_4o_mini_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning GPT-4o-mini in Our Project

We fine-tune `gpt-4o-mini` on Q&A pairs derived from the AI-tutor knowledge base, compare it against GPT-4o and the untuned model inside the same RAG pipeline, and then run the whole fine-tuning lifecycle in code — upload → job → monitor → use → delete — with the OpenAI API.

🖥️ *Runs in **Google Colab or locally** — the only key needed is `OPENAI_API_KEY` (Colab Secrets 🔑, an environment variable, or a `.env` file).*

In [1]:
%pip install -q openai==2.46.0 chromadb==1.5.9 tiktoken==0.13.0 jsonlines==4.0.0 huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

# API key: Colab Secrets (🔑 icon) in Colab; env var or a .env file locally.
if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
elif not os.environ.get("OPENAI_API_KEY"):
    try:  # optional: pick up a local .env if python-dotenv is installed
        from dotenv import load_dotenv

        load_dotenv()
    except ImportError:
        pass

assert os.environ.get("OPENAI_API_KEY"), (
    "Set OPENAI_API_KEY — Colab Secrets in Colab; export it or put it in a .env file locally."
)
print(f"✅ Key loaded — running in {'Colab' if IN_COLAB else 'local'} environment")

✅ Key loaded — running in local environment


## Load the Prebuilt Vector Store

We load the course's **prebuilt** vector store — the AI-tutor knowledge base, already chunked and embedded with `text-embedding-3-small` — from the Towards AI org dataset on Hugging Face: download once, unzip, open. The paths are relative, so this works the same in Colab and locally, and every cell is safe to re-run.

In [ ]:
# Download the prebuilt vector store from the Hugging Face hub (course org dataset)
import pathlib

from huggingface_hub import hf_hub_download

vectorstore = hf_hub_download(
    repo_id="towardsai-tutors/full-stack-ai-engineering-data",
    filename="vector_stores/ai_tutor_knowledge-text-embedding-3-small-1536d-0e801a3f.zip",
    repo_type="dataset",
    local_dir=".",
)
print(vectorstore)

vector_stores/ai_tutor_knowledge-text-em(…): reconstructing file:   0%|          |  0.00B / 97.4MB            

vector_stores/ai_tutor_knowledge-text-em(…): downloading bytes:           |  0.00B            

In [ ]:
import zipfile

STORE_DIR = pathlib.Path("ai_tutor_knowledge-text-embedding-3-small-1536d")

if not STORE_DIR.exists():  # extract once; safe to re-run
    with zipfile.ZipFile(vectorstore) as zf:
        zf.extractall(".")
print("Vector store ready at", STORE_DIR.resolve())

In [ ]:
# Setup an Embedding Model (plain OpenAI SDK — must match the store: text-embedding-3-small, 1536d)

from openai import OpenAI

client = OpenAI()


def embed_query(text):
    return client.embeddings.create(model="text-embedding-3-small", input=text).data[0].embedding

In [ ]:
import chromadb

# Open the prebuilt Chroma collection (the AI-tutor knowledge base, embedded once)
db = chromadb.PersistentClient(path=str(STORE_DIR / "chroma"))
chroma_collection = db.get_collection("ai_tutor_knowledge")
print(chroma_collection.count(), "chunks")

In [ ]:
# The RAG query, written out (replaces the LlamaIndex query engine): embed the
# question, retrieve the top chunks from Chroma, and ask the chosen OpenAI model.


def ask(question, model, top_k=5, temperature=1):
    result = chroma_collection.query(query_embeddings=[embed_query(question)], n_results=top_k)
    sources = [
        {"id": id_, "text": text, "score": 1 - dist, "metadata": meta}
        for id_, text, dist, meta in zip(
            result["ids"][0], result["documents"][0], result["distances"][0], result["metadatas"][0]
        )
    ]
    context = "\n\n---\n\n".join(src["text"] for src in sources)
    completion = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": "Answer the question using only the provided context."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ],
    )
    return completion.choices[0].message.content, sources

## Response Generation Using GPT-4o

In [ ]:
# Query (RAG): retrieve the top-5 chunks, then answer with GPT-4o
response_gpt_4o, sources_gpt_4o = ask(
    "Compare the knowledge retention abilities of a RAG model versus a BERT-based model that has been extensively fine-tuned using PEFT techniques. How do their outputs differ when the knowledge source is removed?",
    model="gpt-4o",
)

response_gpt_4o

In [ ]:
for src in sources_gpt_4o:
    print("Chunk ID\t", src["id"])
    print("Title\t", src["metadata"]["title"])
    print("Text\t", src["text"])
    print("Score\t", round(src["score"], 4))
    print("Metadata\t", src["metadata"])
    print("-_" * 20)

## Response Generation Using GPT-4o-mini

In [ ]:
# Query (RAG): same retrieval, answered with GPT-4o-mini
response_gpt_4o_mini, sources_gpt_4o_mini = ask(
    "Compare the knowledge retention abilities of a RAG model versus a BERT-based model that has been extensively fine-tuned using PEFT techniques. How do their outputs differ when the knowledge source is removed?",
    model="gpt-4o-mini",
)

response_gpt_4o_mini

In [ ]:
for src in sources_gpt_4o_mini:
    print("Chunk ID\t", src["id"])
    print("Title\t", src["metadata"]["title"])
    print("Text\t", src["text"])
    print("Score\t", round(src["score"], 4))
    print("Metadata\t", src["metadata"])
    print("-_" * 20)

## Dataset Format Validation & Number of Tokens in Training Data

In [ ]:
# Format error checks

# https://cookbook.openai.com/examples/chat_finetuning_data_prep

from collections import defaultdict
format_errors = defaultdict(int)

def validate_dataset(output_data):

  for ex in output_data:
      if not isinstance(ex, dict):
          format_errors["data_type"] += 1
          continue

      messages = ex.get("messages", None)
      if not messages:
          format_errors["missing_messages_list"] += 1
          continue

      for message in messages:
          if "role" not in message or "content" not in message:
              format_errors["message_missing_key"] += 1

          if any(k not in ("role", "content", "name", "function_call", "weight") for k in message):
              format_errors["message_unrecognized_key"] += 1

          if message.get("role", None) not in ("system", "user", "assistant", "function"):
              format_errors["unrecognized_role"] += 1

          content = message.get("content", None)
          function_call = message.get("function_call", None)

          if (not content and not function_call) or not isinstance(content, str):
              format_errors["missing_content"] += 1

      if not any(message.get("role", None) == "assistant" for message in messages):
          format_errors["example_missing_assistant_message"] += 1

  if format_errors:
      print("Found errors:")
      for k, v in format_errors.items():
          print(f"{k}: {v}")
  else:
      print("\nNo errors found in the Formatted dataset \n")

In [ ]:
import tiktoken

def counting_no_tokens(output_data):

  tokenizer = tiktoken.encoding_for_model("gpt-4o-mini")

  total_tokens = sum(len(tokenizer.encode(" ".join(message['content'] for message in entry['messages']))) for entry in output_data)

  print(f"Total number of tokens in the Dataset: {total_tokens} \n")

In [ ]:
from huggingface_hub import hf_hub_download
import json
import jsonlines
from pprint import pprint

def dataset_preparation(file_name):
    file_path = hf_hub_download(
        repo_id="jaiganesan/GPT_4o_mini_Fine_tune",
        filename=file_name,
        repo_type="dataset",
        local_dir="."
    )

    with open(file_path, "r") as file:
        data = [json.loads(line) for line in file]

    print("Total entries in the dataset:", len(data))
    print("-_"*30)
    print(data[4])

    output_data = []

    for entry in data:
        formatted_entry = {
            "messages": [
                {"role": "system", "content": "As AI Tutor, answer questions related to AI topics in an in-depth and factual manner."},
                {"role": "user", "content": entry['question']},
                {"role": "assistant", "content": entry['answer']}
            ]
        }
        output_data.append(formatted_entry)

    # Validate and analyze the output data
    validate_dataset(output_data)
    counting_no_tokens(output_data)

    print("-_"*30)
    print(output_data[4])

    base_file_name = os.path.splitext(file_name)[0]
    output_file_path = f'formatted_{base_file_name}.jsonl'

    with jsonlines.open(output_file_path, mode='w') as writer:
        writer.write_all(output_data)

    print(f"\nFormatted dataset has been saved to {output_file_path}.")


In [ ]:
# Training Dataset
dataset_preparation("question_answers_data_100.jsonl")

question_answers_data_100.jsonl:   0%|          | 0.00/276k [00:00<?, ?B/s]

Total entries in the dataset: 100
-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_
{'source': 'tai_blog', 'question': 'What are the key advantages of using BiFPN in object detection compared to conventional methods?', 'answer': "BiFPN, or Bi-directional Feature Pyramid Network, offers several advantages in object detection when compared to conventional methods. It's part of the EfficientDet family of object detectors developed by Google Research and is designed to enhance the efficiency and scalability of object detection models.\n\n### Key Advantages of BiFPN:\n\n1. **Weighted Feature Fusion:**\n   Unlike conventional methods that simply sum up input features during feature fusion, BiFPN introduces learnable weights to adjust the importance of different input features. This means that during multi-scale fusion, input features are not merely combined indiscriminately but are weighted according to their relevance, which enhances the accuracy of the fusion process.\n\n2. **Bi

In [ ]:
# Evaluation Dataset
dataset_preparation("question_answers_data_30.jsonl")

question_answers_data_30.jsonl:   0%|          | 0.00/81.6k [00:00<?, ?B/s]

Total entries in the dataset: 30
-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_
{'source': 'openai_cookbooks', 'question': 'How can creating high-quality evaluations for large language models like GPT-4 improve the stability and reliability of AI applications?', 'answer': "Creating high-quality evaluations for large language models (LLMs), like GPT-4, significantly enhances the stability and reliability of AI applications. Evaluations serve as a robust mechanism to monitor and assess how well these models perform across various scenarios, ultimately leading to improvements in model robustness and reliability.\n\nFirstly, high-quality evaluations can help identify and address areas where models may be underperforming. For instance, systematic evaluations can uncover issues such as drifting performance or deteriorating accuracy over time. By regularly evaluating LLMs against a comprehensive set of benchmarks, developers can detect and correct potential degradation in model 

**This Formatted Training and Evaluation Datasets are being used in the OpenAI Models Fine tuning**

**Up until now, we have explored response generation in RAG System using GPT-4o and GPT-4o-mini and Formatting Training data. Moving forward, we will focus on response generation using a newly fine-tuned model. In our lesson, we explored the process of fine-tuning through the OpenAI UI. However, if you want to learn about Fine tuning OpenAI Models using Code, You can explore the code sections in the notebook.**

## Response Generation Using New Fine Tuned Model

In [ ]:
# Fine Tuned Model

FINE_TUNED_MODEL = "ft:gpt-4o-mini-2024-07-18:towards-ai:ai-tutor:AIc6MRSB"

In [ ]:
# Query (RAG): same retrieval, answered with the fine-tuned model
response_fine_tuned_model, sources_fine_tuned_model = ask(
    "Compare the knowledge retention abilities of a RAG model versus a BERT-based model that has been extensively fine-tuned using PEFT techniques. How do their outputs differ when the knowledge source is removed?",
    model=FINE_TUNED_MODEL,
)

response_fine_tuned_model

In [ ]:
for src in sources_fine_tuned_model:
    print("Chunk ID\t", src["id"])
    print("Title\t", src["metadata"]["title"])
    print("Text\t", src["text"])
    print("Score\t", round(src["score"], 4))
    print("Metadata\t", src["metadata"])
    print("-_" * 20)

## Fine-Tuning OpenAI Models Using the OpenAI API (Code)

### Upload the Training File

In [ ]:
from openai import OpenAI
client = OpenAI()

fine_tune_file = client.files.create(
    file=open("formatted_training_data.json", "rb"),
    purpose="fine-tune"
)

In [ ]:
pprint(fine_tune_file)

FileObject(id='file-yjrbFrpGxfte1fIrnwgAX12A', bytes=291689, created_at=1728892283, filename='formatted_training_data.json', object='file', purpose='fine-tune', status='processed', status_details=None)


In [ ]:
param_training_file_name = fine_tune_file.id
pprint(param_training_file_name)

'file-yjrbFrpGxfte1fIrnwgAX12A'


### Create the Fine-Tune Job

In [ ]:
result_job = client.fine_tuning.jobs.create(
    training_file=param_training_file_name,
    model="gpt-4o-mini-2024-07-18",
    hyperparameters={ "n_epochs":2 }
)

pprint(result_job)

In [ ]:
param_file_tune_job_id = result_job.id
pprint(param_file_tune_job_id)

### Monitor the Fine-Tuning Job

In [ ]:
# Retrieve the state of a fine-tune
client.fine_tuning.jobs.retrieve(param_file_tune_job_id)

In [ ]:
# Retrieve the status of a fine-tune
client.fine_tuning.jobs.retrieve(param_file_tune_job_id).status

In [ ]:
import time
from datetime import datetime

while True:
  time.sleep(20)
  try:
      job_status = client.fine_tuning.jobs.retrieve(param_file_tune_job_id)
      print(f"------------ Job Status: {job_status.status} --------------")

      if job_status.status in ["failed", "succeeded", "cancelled"]:
          print("Job Completed. Detailed Events list:")
          events = client.fine_tuning.jobs.list_events(fine_tuning_job_id=param_file_tune_job_id)
          for event in events:
              print(f'{datetime.fromtimestamp(event.created_at)} {event.message}')

          print("######## Fine-tuned model ###########")
          print(f"{job_status.fine_tuned_model}")
          print("#####################################")
          break
  except Exception as e:
      print(f"Error monitoring job: {e}")
      break

In [ ]:
# # Retrieve the state of a fine-tune

# while client.fine_tuning.jobs.retrieve(param_file_tune_job_id).status != 'succeeded':
#   sleep(10)

In [ ]:
# Retrieve the state of a fine-tune
client.fine_tuning.jobs.retrieve(param_file_tune_job_id).status

In [ ]:
param_file_tune_model = client.fine_tuning.jobs.retrieve(param_file_tune_job_id).fine_tuned_model
pprint(param_file_tune_model)

### Deleting a Fine-Tuned Model

In [ ]:
# Delete a fine-tuned model (must be an owner of the org the model was created in)
# result_delete = client.models.delete(param_file_tune_model)
# pprint(result_delete)